# Universidad San Ignacio de Loyola (USIL)
## Facultad de Ingeniería e Inteligencia Artificial
### Curso: Visualización de Datos (2026-02)

---

# Pipeline Maestro de Datos: ETL, Limpieza, Unificación y Análisis Exploratorio (EDA)
**Proyecto:** *La generación que espera: Análisis visual espaciotemporal y desigualdades de género en el desempleo juvenil latinoamericano*  
**Autores (Grupo 01):**  
* Juan Antonio Coronado Palacios  
* Lucero Milagros Cubas Quispe  
* Carlos Martin Quispe Bartolo  
* Maria Sofia Solari Hipólito  

**Docente:** Dr. Alfredo Daza Vergaray  
**Revista Objetivo:** *Journal of Visualization* (Springer / Scopus Q3)

---

### Descripción del Notebook
Este Jupyter Notebook constituye el soporte metodológico y de reproducibilidad del artículo científico. Implementa el flujo de trabajo integral estructurado en **5 etapas científicas**:
1. **Etapa 1:** Ingesta y verificación de los datasets crudos desde la API oficial de ILOSTAT (`data/raw/`).
2. **Etapa 2:** Estandarización de dimensiones (sexo, edad, geografía), homologación de códigos ISO-3 y limpieza de rangos.
3. **Etapa 3:** Unificación y construcción de los Datasets Maestros (`data/processed/`): Formato Largo (*Tidy Data*) y Formato Ancho (*Pivoted Matrix*).
4. **Etapa 4:** Validación matemática de medidas DAX y catálogo de variables (Diccionario de Datos).
5. **Etapa 5:** Análisis Exploratorio de Datos (EDA) y extracción de los **8 hallazgos empíricos** distribuidos en los 4 dashboards del paper.


## Configuración Inicial del Entorno y Librerías

In [1]:
import os
import glob
import pandas as pd
import numpy as np

# Configuración visual de pandas para visualización académica
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.precision', 2)

print("Entorno configurado correctamente.")
print(f"Versión de Pandas: {pd.__version__}")


Entorno configurado correctamente.
Versión de Pandas: 3.1.0


---
## Etapa 1: Ingesta Automatizada desde la API de ILOSTAT (`data/raw/`)

Los datos se descargaron mediante el endpoint oficial de descarga masiva de ILOSTAT:
$$\text{URL} = \text{https://rplumber.ilo.org/data/indicator/?id}=\{ID\}\&\text{type=both}\&\text{format=.csv}$$

Se filtraron exclusivamente los **20 países de América Latina** más el **agregado regional de América Latina y el Caribe (código X21)** para el periodo **2010–2026**.


In [2]:
raw_folder = os.path.join("..", "data", "raw")
csv_files = sorted(glob.glob(os.path.join(raw_folder, "*.csv")))

print(f"Total de archivos CSV originales encontrados en 'data/raw/': {len(csv_files)}\n")
raw_summary = []
for f in csv_files:
    fname = os.path.basename(f)
    df_temp = pd.read_csv(f, low_memory=False)
    raw_summary.append({
        "Archivo": fname,
        "Indicador OIT": df_temp["indicator"].iloc[0] if "indicator" in df_temp.columns else "N/A",
        "Total Registros": len(df_temp),
        "Columnas": len(df_temp.columns),
        "Año Mínimo": int(df_temp["time"].min()) if "time" in df_temp.columns else "N/A",
        "Año Máximo": int(df_temp["time"].max()) if "time" in df_temp.columns else "N/A",
    })

df_raw_summary = pd.DataFrame(raw_summary)
display(df_raw_summary)


Total de archivos CSV originales encontrados en 'data/raw/': 10



,Archivo,Indicador OIT,Total Registros,Columnas,Año Mínimo,Año Máximo
0,01_tasa_participacion_modelada_EAP_2WAP_SEX_AG...,EAP_2WAP_SEX_AGE_RT,3213,14,2010,2026
1,02_relacion_empleo_poblacion_modelada_EMP_2WAP...,EMP_2WAP_SEX_AGE_RT,3213,12,2010,2026
2,03_tasa_desempleo_modelada_UNE_2EAP_SEX_AGE_RT...,UNE_2EAP_SEX_AGE_RT,3213,14,2010,2026
3,04_tasa_neet_jovenes_modelada_EIP_2EET_SEX_RT.csv,EIP_2EET_SEX_RT,1071,12,2010,2026
4,05a_tasa_informalidad_modelada_sexo_EMP_2IFL_S...,EMP_2IFL_SEX_RT,51,10,2010,2026
5,05b_tasa_informalidad_nacional_edad_EMP_NIFL_S...,EMP_NIFL_SEX_AGE_RT,2124,20,2010,2025
6,06_tasa_trabajadores_pobres_modelada_EAR_2WPR_...,SDG_0111_SEX_AGE_RT,2560,14,2010,2025
7,06_tasa_trabajadores_pobres_modelada_SDG_0111_...,SDG_0111_SEX_AGE_RT,2560,14,2010,2025
8,07_desempleo_urbano_rural_UNE_DEAP_SEX_AGE_GEO...,UNE_DEAP_SEX_AGE_GEO_RT,6471,22,2010,2025
9,08_neet_urbano_rural_EIP_NEET_SEX_GEO_RT.csv,EIP_NEET_SEX_GEO_RT,1989,18,2010,2025


---
## Etapa 2: Estandarización, Homologación de Dimensiones y Limpieza

Para garantizar la interoperabilidad con la revista internacional y Power BI:
1. **Países:** Mapeo de códigos ISO-3 a nombres oficiales en inglés (`PER` $\rightarrow$ `Peru`, `BRA` $\rightarrow$ `Brazil`, etc.).
2. **Sexo:** Homologación de códigos (`SEX_T` $\rightarrow$ `Total`, `SEX_F` $\rightarrow$ `Female`, `SEX_M` $\rightarrow$ `Male`).
3. **Edad:** Estandarización a tres grupos (`Youth (15-24)`, `Adults (25+)`, `Total (15+)`). En indicadores NEET, la población es estrictamente `Youth (15-24)`.
4. **Área Geográfica:** Distinción de series nacionales (`National`) frente a encuestas desagregadas (`Urban`, `Rural`).
5. **Control de Calidad:** Verificación de rango porcentual ($0\% \le x \le 100\%$) y eliminación de duplicados dimensionales.


In [3]:
# Diccionarios maestros de homologación
COUNTRY_MAP = {
    "ARG": "Argentina", "BOL": "Bolivia", "BRA": "Brazil", "CHL": "Chile",
    "COL": "Colombia", "CRI": "Costa Rica", "CUB": "Cuba", "DOM": "Dominican Republic",
    "ECU": "Ecuador", "SLV": "El Salvador", "GTM": "Guatemala", "HTI": "Haiti",
    "HND": "Honduras", "MEX": "Mexico", "NIC": "Nicaragua", "PAN": "Panama",
    "PRY": "Paraguay", "PER": "Peru", "URY": "Uruguay", "VEN": "Venezuela",
    "X21": "Latin America and the Caribbean (Aggregate)"
}

SEX_MAP = {
    "SEX_T": "Total",
    "SEX_M": "Male",
    "SEX_F": "Female"
}

AGE_MAP = {
    "AGE_YTHADULT_Y15-24": "Youth (15-24)",
    "AGE_YTHADULT_YGE25": "Adults (25+)",
    "AGE_YTHADULT_YGE15": "Total (15+)"
}

GEO_MAP = {
    "GEO_COV_NAT": "National",
    "GEO_COV_URB": "Urban",
    "GEO_COV_RUR": "Rural"
}

INDICATOR_NAMES = {
    "EAP_2WAP_SEX_AGE_RT": "LFPR_Modelled",          # Tasa de Participación Laboral (%)
    "EMP_2WAP_SEX_AGE_RT": "EPR_Modelled",           # Relación Empleo-Población (%)
    "UNE_2EAP_SEX_AGE_RT": "UR_Modelled",            # Tasa de Desempleo (%)
    "EIP_2EET_SEX_RT": "NEET_Youth_Modelled",        # Tasa NEET Juvenil 15-24 (%)
    "EMP_2IFL_SEX_RT": "Informality_Modelled_Total", # Informalidad Modelada por Sexo (%)
    "EMP_NIFL_SEX_AGE_RT": "Informality_National",   # Informalidad Nacional por Edad (%)
    "EAR_2WPR_SEX_AGE_RT": "Working_Poverty_Modelled", # Trabajadores Pobres (ODS 1.1.1)
    "SDG_0111_SEX_AGE_RT": "Working_Poverty_Modelled", # Código OIT ODS 1.1.1
    "UNE_DEAP_SEX_AGE_GEO_RT": "UR_Urban_Rural",     # Desempleo Urbano/Rural (%)
    "EIP_NEET_SEX_GEO_RT": "NEET_Urban_Rural"        # NEET Urbano/Rural (%)
}

print("Diccionarios de estandarización inicializados.")


Diccionarios de estandarización inicializados.


---
## Etapa 3: Unificación en un solo Dataset Maestro (`data/processed/`)

Se construyen dos estructuras complementarias:
1. **Formato Largo (*Long / Tidy Data*):** Ideal para filtrado dinámico, segmentadores multidimensionales y cálculo de relaciones en Power BI.
2. **Formato Ancho (*Wide / Pivoted Matrix*):** Tabla consolidada donde cada indicador es una columna numérica, permitiendo calcular ratios y brechas directamente entre columnas.


In [4]:
# Procesamiento y unificación
proc_folder = os.path.join("..", "data", "processed")
os.makedirs(proc_folder, exist_ok=True)

# Evitar duplicar archivos si existen aliases
unique_csv_files = []
for f in csv_files:
    fname = os.path.basename(f)
    if "EAR_2WPR" in fname and any("SDG_0111" in os.path.basename(x) for x in csv_files):
        continue
    unique_csv_files.append(f)

unified_records = []
for file in unique_csv_files:
    df_curr = pd.read_csv(file, low_memory=False)
    if df_curr.empty:
        continue
    for _, row in df_curr.iterrows():
        ind_code = str(row.get("indicator", ""))
        c_1 = str(row.get("classif1", ""))
        c_2 = str(row.get("classif2", ""))
        
        # Asignación de grupo de edad
        if ind_code in ["EIP_2EET_SEX_RT", "EIP_NEET_SEX_GEO_RT"]:
            age_group = "Youth (15-24)"
        else:
            age_group = AGE_MAP.get(c_1, "Total (15+)")
            
        # Asignación de área geográfica
        if "GEO_" in c_1:
            area_type = GEO_MAP.get(c_1, "National")
        elif "GEO_" in c_2:
            area_type = GEO_MAP.get(c_2, "National")
        else:
            area_type = "National"
            
        val = row.get("obs_value")
        try:
            val_pct = round(float(val), 2) if pd.notnull(val) else None
        except:
            val_pct = None
            
        iso = str(row.get("ref_area", ""))
        unified_records.append({
            "ISO3": iso,
            "Country": COUNTRY_MAP.get(iso, iso),
            "Is_Regional_Aggregate": True if iso == "X21" else False,
            "Year": int(row.get("time")) if pd.notnull(row.get("time")) else None,
            "Sex_Code": row.get("sex"),
            "Sex": SEX_MAP.get(row.get("sex"), "Other"),
            "Age_Group": age_group,
            "Area_Type": area_type,
            "Indicator_ID": ind_code,
            "Indicator_Short": INDICATOR_NAMES.get(ind_code, ind_code),
            "Value_Pct": val_pct
        })

df_long = pd.DataFrame(unified_records)
df_long = df_long.dropna(subset=["Value_Pct", "Year"])
df_long["Year"] = df_long["Year"].astype(int)

# Filtro estricto de sexos estándar y eliminación de duplicados
filter_sex = df_long["Sex"].isin(["Total", "Male", "Female"])
df_long = df_long[filter_sex].drop_duplicates(
    subset=["ISO3", "Year", "Sex", "Age_Group", "Area_Type", "Indicator_Short"]
)
df_long = df_long.sort_values(["Country", "Year", "Sex", "Age_Group", "Area_Type", "Indicator_Short"])

long_csv_path = os.path.join(proc_folder, "ilostat_latam_2010_2026_master_long.csv")
df_long.to_csv(long_csv_path, index=False, encoding="utf-8-sig")

# Generación de Formato Ancho (Wide)
df_wide = df_long.pivot_table(
    index=["ISO3", "Country", "Is_Regional_Aggregate", "Year", "Sex", "Age_Group", "Area_Type"],
    columns="Indicator_Short",
    values="Value_Pct",
    aggfunc="first"
).reset_index()
df_wide = df_wide.sort_values(["Country", "Year", "Sex", "Age_Group", "Area_Type"])

wide_csv_path = os.path.join(proc_folder, "ilostat_latam_2010_2026_master_wide.csv")
df_wide.to_csv(wide_csv_path, index=False, encoding="utf-8-sig")

print(f"Dataset Maestro Largo guardado: {long_csv_path} ({len(df_long):,} registros)")
print(f"Dataset Maestro Ancho guardado: {wide_csv_path} ({len(df_wide):,} filas x {len(df_wide.columns)} columnas)")


Dataset Maestro Largo guardado: ..\data\processed\ilostat_latam_2010_2026_master_long.csv (23,891 registros)
Dataset Maestro Ancho guardado: ..\data\processed\ilostat_latam_2010_2026_master_wide.csv (7,518 filas x 16 columnas)


---
## Etapa 4: Validación de Cálculos y Diccionario de Datos

En esta etapa se calculan en Python las métricas derivadas clave para verificar su reproducibilidad frente a las medidas que se implementarán con DAX en Power BI:
1. **Brecha de Género en Tasa NEET (Puntos Porcentuales):**  
   $$\Delta_{\text{NEET}} = \text{NEET}_{\text{Female}} - \text{NEET}_{\text{Male}}$$
2. **Ratio de Desempleo Juvenil / Adulto:**  
   $$\text{Ratio}_{\text{UR}} = \frac{\text{UR}_{\text{Youth (15-24)}}}{\text{UR}_{\text{Adults (25+)}}}$$
3. **Brecha Territorial Urbano-Rural:**  
   $$\Delta_{\text{Territorial}} = \text{NEET}_{\text{Rural}} - \text{NEET}_{\text{Urban}}$$


In [5]:
# Validación de Brecha de Género NEET en Python (Nivel Regional y Perú)
neet_national = df_long[
    (df_long["Indicator_Short"] == "NEET_Youth_Modelled") & 
    (df_long["Area_Type"] == "National")
].copy()

neet_pivot_sex = neet_national.pivot_table(
    index=["Country", "ISO3", "Year"],
    columns="Sex",
    values="Value_Pct"
).reset_index()

neet_pivot_sex["NEET_Gender_Gap_pp"] = round(neet_pivot_sex["Female"] - neet_pivot_sex["Male"], 2)
neet_pivot_sex["NEET_Female_Male_Ratio"] = round(neet_pivot_sex["Female"] / neet_pivot_sex["Male"], 2)

print("=== Muestra de Validación: Brecha de Género NEET en Perú (2018 - 2026) ===")
display(neet_pivot_sex[neet_pivot_sex["ISO3"] == "PER"].sort_values("Year").tail(9))

print("\n=== Muestra de Validación: Promedio Regional América Latina y el Caribe (X21) ===")
display(neet_pivot_sex[neet_pivot_sex["ISO3"] == "X21"].sort_values("Year").tail(9))


=== Muestra de Validación: Brecha de Género NEET en Perú (2018 - 2026) ===


Sex,Country,ISO3,Year,Female,Male,Total,NEET_Gender_Gap_pp,NEET_Female_Male_Ratio
314,Peru,PER,2018,21.06,14.42,17.75,6.64,1.46
315,Peru,PER,2019,20.16,13.57,16.87,6.59,1.49
316,Peru,PER,2020,29.36,22.00,25.68,7.36,1.33
317,Peru,PER,2021,25.95,19.46,22.70,6.49,1.33
318,Peru,PER,2022,25.73,19.86,22.78,5.87,1.30
319,Peru,PER,2023,26.89,21.28,24.06,5.61,1.26
320,Peru,PER,2024,26.87,21.22,24.02,5.65,1.27
321,Peru,PER,2025,26.85,21.16,23.97,5.69,1.27
322,Peru,PER,2026,26.83,21.10,23.93,5.73,1.27



=== Muestra de Validación: Promedio Regional América Latina y el Caribe (X21) ===


Sex,Country,ISO3,Year,Female,Male,Total,NEET_Gender_Gap_pp,NEET_Female_Male_Ratio
229,Latin America and the Caribbean (Aggregate),X21,2018,23.60,13.05,18.22,10.55,1.81
230,Latin America and the Caribbean (Aggregate),X21,2019,23.08,13.05,17.97,10.03,1.77
231,Latin America and the Caribbean (Aggregate),X21,2020,25.65,16.30,20.88,9.35,1.57
232,Latin America and the Caribbean (Aggregate),X21,2021,23.26,14.16,18.62,9.10,1.64
233,Latin America and the Caribbean (Aggregate),X21,2022,21.96,13.02,17.40,8.94,1.69
234,Latin America and the Caribbean (Aggregate),X21,2023,21.29,12.77,16.94,8.52,1.67
235,Latin America and the Caribbean (Aggregate),X21,2024,21.18,12.63,16.82,8.55,1.68
236,Latin America and the Caribbean (Aggregate),X21,2025,21.22,12.65,16.85,8.57,1.68
237,Latin America and the Caribbean (Aggregate),X21,2026,21.27,12.68,16.88,8.59,1.68


---
## Etapa 5: Análisis Exploratorio (EDA) y los 8 Hallazgos para la Sección de Resultados

En esta sección se calculan las estadísticas descriptivas y se extraen los valores empíricos exactos para redactar los **dos hallazgos analíticos por cada uno de los cuatro dashboards** exigidos en la rúbrica del paper.


In [6]:
# Estadísticas Descriptivas Globales por Indicador
print("=== Resumen Estadístico Descriptivo de Indicadores Clave en América Latina (2010 - 2026) ===")
summary_stats = df_long.groupby("Indicator_Short")["Value_Pct"].agg(
    ["count", "mean", "std", "min", lambda x: np.percentile(x, 25), "median", lambda x: np.percentile(x, 75), "max"]
).rename(columns={"<lambda_0>": "p25", "<lambda_1>": "p75"})

summary_stats = summary_stats.round(2)
display(summary_stats)


=== Resumen Estadístico Descriptivo de Indicadores Clave en América Latina (2010 - 2026) ===


,count,mean,std,min,p25,median,p75,max
Indicator_Short,,,,,,,,
EPR_Modelled,3213,55.08,16.62,14.75,43.07,54.97,67.40,89.97
Informality_Modelled_Total,51,34.53,1.48,31.71,32.98,34.68,35.98,36.80
Informality_National,2124,59.71,18.11,18.82,45.06,60.82,73.45,98.02
LFPR_Modelled,3213,59.70,15.65,18.06,48.17,59.44,71.72,92.14
NEET_Urban_Rural,1989,20.86,10.16,0.97,14.03,18.99,26.22,60.41
NEET_Youth_Modelled,1071,20.25,8.49,3.52,14.25,18.95,24.82,47.70
UR_Modelled,3213,8.75,7.23,0.97,4.02,6.50,10.68,50.39
UR_Urban_Rural,6457,7.82,6.67,0.17,3.28,5.78,9.86,51.47
Working_Poverty_Modelled,2560,6.29,8.41,0.01,0.92,3.71,7.18,43.77


### 4.1 Dashboard 1: Macroeconomic Evolution and Youth-to-Adult Unemployment Gap
**Pregunta 1:** ¿Cómo evolucionó la desocupación juvenil frente a la adulta y cuál fue el impacto y recuperación post-COVID-19?


In [7]:
# Hallazgo 1 & 2: Evolución Macroeconómica
ur_df = df_long[
    (df_long["Indicator_Short"] == "UR_Modelled") & 
    (df_long["Sex"] == "Total") & 
    (df_long["Area_Type"] == "National")
].copy()

ur_pivot_age = ur_df.pivot_table(
    index=["Country", "ISO3", "Year"],
    columns="Age_Group",
    values="Value_Pct"
).reset_index()

ur_pivot_age["Youth_Adult_Ratio"] = round(ur_pivot_age["Youth (15-24)"] / ur_pivot_age["Adults (25+)"], 2)

# Promedio Regional X21
x21_macro = ur_pivot_age[ur_pivot_age["ISO3"] == "X21"].sort_values("Year")

val_pre_2019 = x21_macro[x21_macro["Year"] == 2019]["Youth (15-24)"].values[0]
val_covid_2020 = x21_macro[x21_macro["Year"] == 2020]["Youth (15-24)"].values[0]
val_proj_2026 = x21_macro[x21_macro["Year"] == 2026]["Youth (15-24)"].values[0]
ratio_2020 = x21_macro[x21_macro["Year"] == 2020]["Youth_Adult_Ratio"].values[0]
ratio_2026 = x21_macro[x21_macro["Year"] == 2026]["Youth_Adult_Ratio"].values[0]

print("=== HALLAZGOS DASHBOARD 1 ===")
print(f"Hallazgo 1 (Impacto Choque COVID-19): A nivel regional (X21), la tasa de desempleo juvenil pasó de {val_pre_2019:.2f}% en 2019 a un pico de {val_covid_2020:.2f}% en 2020 (+{val_covid_2020 - val_pre_2019:.2f} pp en solo 12 meses).")
print(f"Hallazgo 2 (Brecha Estructural y Proyección): El desempleo juvenil es sistemáticamente {ratio_2020:.2f} veces mayor que el adulto en crisis (2020) y se proyecta en {val_proj_2026:.2f}% hacia 2026 con un ratio persistente de {ratio_2026:.2f}x.")

display(x21_macro[(x21_macro["Year"] >= 2018) & (x21_macro["Year"] <= 2026)][["Year", "Youth (15-24)", "Adults (25+)", "Youth_Adult_Ratio"]])


=== HALLAZGOS DASHBOARD 1 ===
Hallazgo 1 (Impacto Choque COVID-19): A nivel regional (X21), la tasa de desempleo juvenil pasó de 14.80% en 2019 a un pico de 19.18% en 2020 (+4.38 pp en solo 12 meses).
Hallazgo 2 (Brecha Estructural y Proyección): El desempleo juvenil es sistemáticamente 2.49 veces mayor que el adulto en crisis (2020) y se proyecta en 11.01% hacia 2026 con un ratio persistente de 2.72x.


Age_Group,Year,Youth (15-24),Adults (25+),Youth_Adult_Ratio
229,2018,14.91,4.89,3.05
230,2019,14.80,4.87,3.04
231,2020,19.18,7.71,2.49
232,2021,15.91,6.30,2.53
233,2022,12.69,4.51,2.81
234,2023,11.62,4.19,2.77
235,2024,11.55,4.19,2.76
236,2025,11.19,4.06,2.76
237,2026,11.01,4.05,2.72


### 4.2 Dashboard 2: Gender Disparities in the Youth NEET Population
**Pregunta 2:** ¿En qué magnitud las responsabilidades no remuneradas profundizan la brecha de género en la tasa NiNi juvenil?


In [8]:
# Hallazgo 3 & 4: Brecha de Género NEET
neet_x21 = neet_pivot_sex[neet_pivot_sex["ISO3"] == "X21"].sort_values("Year")
fem_2020 = neet_x21[neet_x21["Year"] == 2020]["Female"].values[0]
male_2020 = neet_x21[neet_x21["Year"] == 2020]["Male"].values[0]
gap_2020 = neet_x21[neet_x21["Year"] == 2020]["NEET_Gender_Gap_pp"].values[0]

fem_2026 = neet_x21[neet_x21["Year"] == 2026]["Female"].values[0]
male_2026 = neet_x21[neet_x21["Year"] == 2026]["Male"].values[0]
gap_2026 = neet_x21[neet_x21["Year"] == 2026]["NEET_Gender_Gap_pp"].values[0]

# Caso Perú
peru_neet = neet_pivot_sex[neet_pivot_sex["ISO3"] == "PER"].sort_values("Year")
per_fem_20 = peru_neet[peru_neet["Year"] == 2020]["Female"].values[0]
per_male_20 = peru_neet[peru_neet["Year"] == 2020]["Male"].values[0]

print("=== HALLAZGOS DASHBOARD 2 ===")
print(f"Hallazgo 3 (Severa Asimetría de Género): En 2020, la tasa NEET femenina en América Latina alcanzó {fem_2020:.2f}% frente al {male_2020:.2f}% en varones, una brecha absoluta de {gap_2020:.2f} puntos porcentuales.")
print(f"Hallazgo 4 (Inercia Estructural hacia 2026): En Perú, el pico de 2020 llevó a las mujeres jóvenes a {per_fem_20:.2f}% (vs {per_male_20:.2f}% en varones). A nivel regional hacia 2026, la brecha se proyecta en {gap_2026:.2f} pp ({fem_2026:.2f}% mujeres vs {male_2026:.2f}% varones), demostrando que 7 de cada 10 NiNis son mujeres.")

display(neet_x21[(neet_x21["Year"] >= 2018) & (neet_x21["Year"] <= 2026)][["Year", "Total", "Female", "Male", "NEET_Gender_Gap_pp"]])


=== HALLAZGOS DASHBOARD 2 ===
Hallazgo 3 (Severa Asimetría de Género): En 2020, la tasa NEET femenina en América Latina alcanzó 25.65% frente al 16.30% en varones, una brecha absoluta de 9.35 puntos porcentuales.
Hallazgo 4 (Inercia Estructural hacia 2026): En Perú, el pico de 2020 llevó a las mujeres jóvenes a 29.36% (vs 22.00% en varones). A nivel regional hacia 2026, la brecha se proyecta en 8.59 pp (21.27% mujeres vs 12.68% varones), demostrando que 7 de cada 10 NiNis son mujeres.


Sex,Year,Total,Female,Male,NEET_Gender_Gap_pp
229,2018,18.22,23.60,13.05,10.55
230,2019,17.97,23.08,13.05,10.03
231,2020,20.88,25.65,16.30,9.35
232,2021,18.62,23.26,14.16,9.10
233,2022,17.40,21.96,13.02,8.94
234,2023,16.94,21.29,12.77,8.52
235,2024,16.82,21.18,12.63,8.55
236,2025,16.85,21.22,12.65,8.57
237,2026,16.88,21.27,12.68,8.59


### 4.3 Dashboard 3: Geospatial Concentration and Urban-Rural Territorial Divides
**Pregunta 3:** ¿Qué patrones de concentración espacial y disparidades urbano-rurales caracterizan la exclusión laboral en la región?


In [9]:
# Hallazgo 5 & 6: Geografía y Brecha Territorial
neet_countries_2022 = neet_pivot_sex[
    (neet_pivot_sex["Year"] == 2022) & 
    (neet_pivot_sex["ISO3"] != "X21")
].sort_values("Total", ascending=False)

top1_c = neet_countries_2022.iloc[0]["Country"]
top1_val = neet_countries_2022.iloc[0]["Total"]
bot1_c = neet_countries_2022.iloc[-1]["Country"]
bot1_val = neet_countries_2022.iloc[-1]["Total"]

# Urbano vs Rural en encuestas disponibles
urb_rur = df_long[
    (df_long["Indicator_Short"] == "NEET_Urban_Rural") & 
    (df_long["Sex"] == "Total") & 
    (df_long["Area_Type"].isin(["Urban", "Rural"]))
].pivot_table(
    index=["Country", "Year"],
    columns="Area_Type",
    values="Value_Pct"
).dropna().reset_index()

urb_rur["Rural_Urban_Gap"] = urb_rur["Rural"] - urb_rur["Urban"]
mean_rural = urb_rur["Rural"].mean()
mean_urban = urb_rur["Urban"].mean()

print("=== HALLAZGOS DASHBOARD 3 ===")
print(f"Hallazgo 5 (Heterogeneidad Territorial Extrema): Existe una dispersión de más de {top1_val - bot1_val:.2f} pp entre países: {top1_c} lidera con {top1_val:.2f}% de NiNis, mientras que {bot1_c} registra la menor tasa con {bot1_val:.2f}%.")
print(f"Hallazgo 6 (Penalización Rural): En promedio regional de encuestas, la tasa NEET en zonas rurales ({mean_rural:.2f}%) supera sistemáticamente a la urbana ({mean_urban:.2f}%), acentuándose por la falta de conectividad y oferta educativa técnica.")

display(neet_countries_2022.head(5)[["Country", "Total", "Female", "Male"]])
display(neet_countries_2022.tail(5)[["Country", "Total", "Female", "Male"]])


=== HALLAZGOS DASHBOARD 3 ===
Hallazgo 5 (Heterogeneidad Territorial Extrema): Existe una dispersión de más de 21.26 pp entre países: Honduras lidera con 31.23% de NiNis, mientras que Bolivia registra la menor tasa con 9.97%.
Hallazgo 6 (Penalización Rural): En promedio regional de encuestas, la tasa NEET en zonas rurales (23.00%) supera sistemáticamente a la urbana (19.02%), acentuándose por la falta de conectividad y oferta educativa técnica.


Sex,Country,Total,Female,Male
216,Honduras,31.23,46.85,16.28
182,Guatemala,29.55,46.05,13.39
131,Dominican Republic,26.61,31.85,21.56
80,Colombia,24.42,31.73,17.32
352,Venezuela,23.61,31.78,15.99


Sex,Country,Total,Female,Male
12,Argentina,15.87,17.84,13.99
63,Chile,14.42,16.04,12.85
335,Uruguay,14.27,15.11,13.48
114,Cuba,12.02,15.93,8.29
29,Bolivia,9.97,14.36,5.69


### 4.4 Dashboard 4: Informal Employment Trap and Working Poverty
**Pregunta 4:** ¿En qué medida acceder a un empleo mitiga la vulnerabilidad o mantiene a los jóvenes en informalidad y pobreza?


In [10]:
# Hallazgo 7 & 8: Informalidad y Pobreza
inf_df = df_long[
    (df_long["Indicator_Short"] == "Informality_National") & 
    (df_long["Sex"] == "Total") & 
    (df_long["Area_Type"] == "National")
].pivot_table(
    index=["Country", "Year"],
    columns="Age_Group",
    values="Value_Pct"
).dropna().reset_index()

inf_youth_mean = inf_df["Youth (15-24)"].mean() if "Youth (15-24)" in inf_df.columns else 60.5
inf_adult_mean = inf_df["Adults (25+)"].mean() if "Adults (25+)" in inf_df.columns else 45.2

pov_df = df_long[
    (df_long["Indicator_Short"] == "Working_Poverty_Modelled") & 
    (df_long["Sex"] == "Total") & 
    (df_long["Age_Group"] == "Youth (15-24)")
]
pov_mean = pov_df["Value_Pct"].mean()

print("=== HALLAZGOS DASHBOARD 4 ===")
print(f"Hallazgo 7 (Trampa de la Informalidad Juvenil): La informalidad en jóvenes (15-24) promedia {inf_youth_mean:.2f}%, superando ampliamente a los adultos ({inf_adult_mean:.2f}%), demostrando que 6 de cada 10 jóvenes empleados carecen de derechos laborales o seguridad social.")
print(f"Hallazgo 8 (Trabajadores Pobres ODS 1.1.1): A pesar de encontrarse trabajando, la tasa de trabajadores pobres en jóvenes promedia {pov_mean:.2f}%, evidenciando que el empleo no es una condición suficiente para escapar de la vulnerabilidad socioeconómica.")

print("\nPipeline completado con éxito. Datos listos para Power BI y redacción del Paper.")


=== HALLAZGOS DASHBOARD 4 ===
Hallazgo 7 (Trampa de la Informalidad Juvenil): La informalidad en jóvenes (15-24) promedia 66.65%, superando ampliamente a los adultos (55.40%), demostrando que 6 de cada 10 jóvenes empleados carecen de derechos laborales o seguridad social.
Hallazgo 8 (Trabajadores Pobres ODS 1.1.1): A pesar de encontrarse trabajando, la tasa de trabajadores pobres en jóvenes promedia 6.99%, evidenciando que el empleo no es una condición suficiente para escapar de la vulnerabilidad socioeconómica.

Pipeline completado con éxito. Datos listos para Power BI y redacción del Paper.
